# Phase 18 — Evaluation harness + SchemaRAG Table 5 ablation

Runs `scripts/run_baseline.py` (CP1 floor) and `scripts/run_eval.py` (full pipeline + 4-way ablation) end-to-end on whatever GPU this session gets. Produces the EX numbers the plan has been waiting on since Phase 15 — nothing here has been run on real held-out data yet, so treat every number this notebook prints as the first look.

SQL uses `Data/cot_data/sql_dev_eval_full.json` — a real held-out Spider **dev**-split set (1034 questions, live SchemaLinker `key_fields`, built in Phase 15A). NoSQL has no native held-out split, so per the README it evaluates on `Data/cot_data/nosql_cot_train.json` — the Generator was fine-tuned on that file, so the NoSQL EX number measures memorization, not generalization. Report it as such in Phase 19.

## 0. Check GPU

In [1]:
import torch

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        total_gb = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
        print(f"GPU {i}: {name}  ({total_gb:.1f} GB)")
        # Mirrors the 20GiB cut in src/generator/infer.py -- keep the two in sync.
        if total_gb >= 20:
            print("  -> >=20GiB (A100/L4-class). GeneratorInfer loads full bf16, no quantization.")
        else:
            print("  -> <20GiB (T4-class). GeneratorInfer loads int8 via bitsandbytes.")
else:
    raise RuntimeError("No GPU visible -- set Runtime > Change runtime type > GPU before continuing.")

GPU 0: NVIDIA A100-SXM4-40GB  (39.5 GB)
  -> >=20GiB (A100/L4-class). GeneratorInfer loads full bf16, no quantization.


## 1. Clone repo + install dependencies

In [2]:
!git clone https://github.com/kethansplunk/Codegen.git
%cd Codegen
!pip install -q torch transformers peft sqlparse pyyaml FlagEmbedding chromadb openai python-dotenv pymongo bitsandbytes

Cloning into 'Codegen'...
remote: Enumerating objects: 1207, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 1207 (delta 18), reused 15 (delta 15), pack-reused 1177 (from 2)
Receiving objects: 100% (1207/1207), 17.90 MiB | 30.14 MiB/s, done.
Resolving deltas: 100% (877/877), done.
/content/Codegen
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 132.9 MB/s eta 0:00:

## 2. Mount Drive and load checkpoints (both tracks)

Loads SAR + Generator checkpoints for **both** SQL and NoSQL from Drive. If your Drive layout differs from `codegen/checkpoints/{sar,generator}_{sql,nosql}`, adjust `DRIVE` below.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/codegen'
os.makedirs('models', exist_ok=True)

for name in ['sar_sql', 'generator_sql', 'sar_nosql', 'generator_nosql']:
    dst = f'models/{name}'
    if not os.path.exists(dst):
        os.symlink(f'{DRIVE}/checkpoints/{name}', dst)

!ls -la models/sar_sql models/generator_sql models/sar_nosql models/generator_nosql

Mounted at /content/drive
lrwxrwxrwx 1 root root 58 Jul 26 08:54 models/generator_nosql -> /content/drive/MyDrive/codegen/checkpoints/generator_nosql
lrwxrwxrwx 1 root root 56 Jul 26 08:54 models/generator_sql -> /content/drive/MyDrive/codegen/checkpoints/generator_sql
lrwxrwxrwx 1 root root 52 Jul 26 08:54 models/sar_nosql -> /content/drive/MyDrive/codegen/checkpoints/sar_nosql
lrwxrwxrwx 1 root root 50 Jul 26 08:54 models/sar_sql -> /content/drive/MyDrive/codegen/checkpoints/sar_sql


In [4]:
# Safety net: force sar.backend to memory. ChromaDB's PersistentClient can't open
# an index over a Google Drive FUSE mount, so this avoids that failure mode entirely.
text = open('configs/config.yaml').read()
text = text.replace('backend: chroma', 'backend: memory')
open('configs/config.yaml', 'w').write(text)
!grep -A1 "^sar:" configs/config.yaml | head -3

sar:
  # "memory" → SARRetriever: re-encodes corpus at startup (~30 sec). No ChromaDB needed.


## 3. Spider SQLite databases

Needed for SQL EX scoring (executing gold + predicted queries), and as the source data for the MongoDB conversion below. Uploaded once as a zip to Drive.

In [5]:
!cp /content/drive/MyDrive/codegen/checkpoints/spider_database.zip /content/Codegen/
!unzip -q /content/Codegen/spider_database.zip -d /content/Codegen/Data/Spider/
!ls /content/Codegen/Data/Spider/database | wc -l

166


### 3b. Data fix — `flight_2` whitespace bug

`Flights.SourceAirport`/`DestAirport` are stored with a leading space (`' APG'`) while `Airports.AirportCode` and every literal in gold SQL aren't (`"APG"`). Raw equality (`WHERE SourceAirport = "APG"`, or any `JOIN ... ON SourceAirport = AirportCode`) silently returns 0 rows instead of erroring, so **gold itself returns the wrong (empty) answer** for any question that filters/joins on these columns — the model isn't being scored against ground truth there, it's being scored against a data bug.

Confirmed by scanning all 166 Spider databases: this is the only one where the padding lands on columns the held-out dev set actually filters/joins on. Affects 44/1034 dev questions (4.3%), 42 of which were silently wrong before the fix. Full diagnosis in this session's conversation history if you need to re-derive it.

Fixes the data in place with `TRIM()` — no model/training changes, this only affects execution-based scoring.

In [ ]:
import sqlite3

path = "Data/Spider/database/flight_2/flight_2.sqlite"
conn = sqlite3.connect(path)
cur = conn.cursor()

fixes = [
    ("airports", "City"), ("airports", "AirportName"), ("airports", "Country"),
    ("airports", "CountryAbbrev"), ("flights", "SourceAirport"), ("flights", "DestAirport"),
]
for table, col in fixes:
    before = cur.execute(f'SELECT COUNT(*) FROM "{table}" WHERE "{col}" <> TRIM("{col}")').fetchone()[0]
    cur.execute(f'UPDATE "{table}" SET "{col}" = TRIM("{col}") WHERE "{col}" <> TRIM("{col}")')
    print(f"{table}.{col}: fixed {before} padded rows")

conn.commit()
conn.close()

# Sanity check. Before the fix this returned 100/100 (every airport, vacuously --
# the join never matched anything, so NOT IN excluded nothing). After the fix it
# should return a small number, not the full 100 -- confirming the filter is now
# actually filtering instead of silently matching everyone.
conn = sqlite3.connect(path)
n = len(conn.execute(
    "SELECT AirportCode FROM Airports WHERE AirportCode NOT IN "
    "(SELECT SourceAirport FROM Flights UNION SELECT DestAirport FROM Flights)"
).fetchall())
total = conn.execute("SELECT COUNT(*) FROM Airports").fetchone()[0]
print(f"\nairports with no flights in/out: {n}/{total} (sanity check -- was 100/100 before the fix)")
conn.close()

## 4. MongoDB setup (needed for NoSQL EX)

**Important**: `Data/mongodb/*.json` schema-cache files are git-tracked from an earlier run on a different machine. `convert_all()` treats their existence as "already converted" and will silently skip real data insertion into this fresh session's empty `mongod` if we don't clear them first. Skip this section if you're only running the SQL track.

In [6]:
# Install and start mongod
!apt-get install -y mongodb >/dev/null 2>&1 || (curl -fsSL https://pgp.mongodb.com/server-7.0.asc | sudo gpg -o /usr/share/keyrings/mongodb-server-7.0.gpg --dearmor && echo "deb [signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg arch=amd64] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse" | sudo tee /etc/apt/sources.list.d/mongodb-org-7.0.list && apt-get update -qq && apt-get install -y mongodb-org)
!mkdir -p /data/db
import subprocess, time
subprocess.Popen(['mongod', '--dbpath', '/data/db', '--logpath', '/var/log/mongod.log', '--fork'])
time.sleep(3)
!tail -n 5 /var/log/mongod.log

deb [signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg arch=amd64] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  mongodb-database-tools mongodb-mongosh mongodb-org-database
  mongodb-org-database-tools-extra mongodb-org-mongos mongodb-org-server
  mongodb-org-shell mongodb-org-tools
The following NEW packages will be installed:
  mongodb-database-tools mongodb-mongosh mongodb-org mongodb-org-database
  mongodb-org-database-tools-extra mongodb-org-mongos mongodb-org-server
  mongodb-org-shell mongodb-org-tools
0 upgraded, 9 newly installed, 0 to remove and 163 not upgraded.
Need to get 189 MB of archives.
After this operation,

In [7]:
import shutil
shutil.rmtree('Data/mongodb', ignore_errors=True)   # force a real reconversion, see note above

from src.mongodb_converter import convert_all
convert_all(
    db_root="Data/Spider/database",
    fk_graph_dir="Data/fk_graphs",
    schema_cache_dir="Data/mongodb",
)

[1/166] academic — 15 collections, 42 fields
[2/166] activity_1 — 5 collections, 22 fields
[3/166] aircraft — 5 collections, 28 fields
[4/166] allergy_1 — 3 collections, 12 fields
[5/166] apartment_rentals — 6 collections, 31 fields
[6/166] architecture — 3 collections, 17 fields
[7/166] assets_maintenance — 14 collections, 64 fields
[8/166] baseball_1 — 26 collections, 352 fields
[9/166] battle_death — 3 collections, 18 fields
[10/166] behavior_monitoring — 11 collections, 64 fields
[11/166] bike_1 — 4 collections, 46 fields
[12/166] body_builder — 2 collections, 11 fields
[13/166] book_2 — 2 collections, 9 fields
[14/166] browser_web — 3 collections, 11 fields
[15/166] candidate_poll — 2 collections, 14 fields
[16/166] car_1 — 6 collections, 23 fields
[17/166] chinook_1 — 11 collections, 64 fields
[18/166] cinema — 3 collections, 17 fields
[19/166] city_record — 4 collections, 27 fields
[20/166] climbing — 2 collections, 12 fields
[21/166] club_1 — 3 collections, 15 fields
[22/166] c

In [8]:
# Verify real data landed (not just schema cache) before trusting any NoSQL EX result
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017")
print(len(client.list_database_names()), client.list_database_names()[:10])
print("formula_1.drivers count:", client['formula_1']['drivers'].count_documents({}))   # must be > 0

169 ['academic', 'activity_1', 'admin', 'aircraft', 'allergy_1', 'apartment_rentals', 'architecture', 'assets_maintenance', 'baseball_1', 'battle_death']
formula_1.drivers count: 842


## 5. DeepSeek API key

Needed by the `SchemaLinker` call inside every `run_eval.py` / `run_baseline.py` invocation below. Paste your real key in place of the placeholder, then re-run this cell — it will not overwrite a key that's already there. **Never commit this cell with a real key filled in.**

In [9]:
from pathlib import Path

PLACEHOLDER = 'your_actual_key_here'
env = Path('.env')
existing = env.read_text() if env.exists() else ''

if 'DEEPSEEK_API_KEY=' in existing and PLACEHOLDER not in existing:
    print('.env already contains a DEEPSEEK_API_KEY -- left untouched.')
else:
    env.write_text(f'DEEPSEEK_API_KEY={PLACEHOLDER}\n')
    print('Wrote .env with a placeholder. Edit it (or this cell) with your real key, then re-run.')

Wrote .env with a placeholder. Edit it (or this cell) with your real key, then re-run.


## 6. Pull the held-out SQL eval set from Drive

`sql_dev_eval_full.json` (1034 Spider dev questions, real SchemaLinker `key_fields`) was built locally in Phase 15A. Upload it to Drive from your Mac first if it isn't there already.

In [18]:
!cp /content/drive/MyDrive/codegen/sql_dev_eval_full.json Data/cot_data/sql_dev_eval_full.json
!wc -l Data/cot_data/sql_dev_eval_full.json

cp: cannot stat '/content/drive/MyDrive/codegen/sql_dev_eval_full.json': No such file or directory
wc: Data/cot_data/sql_dev_eval_full.json: No such file or directory


## 7. SQL track — smoke test before the full run

Timed on a small `--n` so you can extrapolate wall time / compute-unit cost to the full sweep before committing budget to it: `(this cell's wall time / 10) * 100 * 3` (3 generation passes per question for the 4-way ablation, per `plan_generation_passes()`).

In [10]:
%%time
!python -m scripts.run_eval --track sql --data Data/cot_data/sql_dev_eval_full.json \
    --n 10 --ablation all --out evaluation/results/phase18_sql_smoke.json

Evaluating 10 questions x 4 ablation(s) = 30 generation passes (3 per question — configurations sharing generator inputs are batched).
[1/10] What are the name, independence year, and surface area of the country with the smallest population?  (db=world_1)
Running on: cuda
config.json: 100% 779/779 [00:00<00:00, 4.78MB/s]
tokenizer_config.json: 100% 366/366 [00:00<00:00, 2.18MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 13.3MB/s]
tokenizer.json: 100% 711k/711k [00:00<00:00, 50.0MB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 1.02MB/s]

model.safetensors: downloading bytes:  12% 156M/1.34G [00:01<00:05, 236MB/s, 12.1MB/s  ]
model.safetensors: downloading bytes:  15% 201M/1.34G [00:01<00:04, 231MB/s, 17.0MB/s  ]
model.safetensors: downloading bytes:  20% 269M/1.34G [00:01<00:03, 277MB/s, 22.3MB/s  ]
model.safetensors: downloading bytes:  41% 556M/1.34G [00:02<00:01, 395MB/s, 46.0MB/s  ]
model.safetensors: downloading bytes:  47% 635M/1.34G [00:02<00:01, 395MB/s, 53.3MB/s  ]
mode

## 8. SQL track — full Table 5 ablation sweep (the Phase 18 headline numbers)

`full` / `no_schema_linker` / `no_sar` / `no_posg` on 100 held-out Spider dev questions. Targets: **>82% EX** for `full`, with `no_schema_linker` / `no_sar` / `no_posg` each expected to fall below it -- that gap is Table 5.

In [22]:
%%time
!python -m scripts.run_eval --track sql --data Data/cot_data/sql_dev_eval_full.json \
    --n 100 --ablation all --out evaluation/results/phase18_sql_full.json

Evaluating 100 questions x 4 ablation(s) = 300 generation passes (3 per question — configurations sharing generator inputs are batched).
[1/100] What are the name, independence year, and surface area of the country with the smallest population?  (db=world_1)
Running on: cuda
Loading weights: 100% 391/391 [00:00<00:00, 2835.47it/s]
Loaded corpus: 7000 entries
Pre-computing corpus embeddings ...
pre tokenize: 100% 28/28 [00:00<00:00, 215.57it/s]
Inference Embeddings: 100% 28/28 [00:00<00:00, 32.92it/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:03<00:00, 87.93it/s] 
    full              ex=1.0  exact=False
    no_schema_linker  ex=1.0  exact=False
    no_sar            ex=1.0  exact=False
    no_posg           ex=1.0  exact=False
[2/100] What are years of founding for orchestras that have had more than a single performance?  (db=orchestra)
    full              ex=1.0  exact=False
    no_schema_linker  ex=1.0  exact=False
    no_sa

## 9. Why is `full` at 79.0% instead of >82%? — failure breakdown

Pure local analysis of the report written in section 9 — no GPU/model calls, so this is free to re-run. Buckets every scored question by the same `_is_hard()` heuristic `--hard` uses (multi-JOIN / subquery / GROUP BY) and prints the actual failing question/gold/predicted query for anything `full` got wrong, so you can eyeball whether the shortfall concentrates in complex queries or is spread evenly.## 9b. Why is `full` at 79.0% instead of >82%? — failure breakdown

Pure local analysis of the report written in section 9 — no GPU/model calls, so this is free to re-run. Buckets every scored question by the same `_is_hard()` heuristic `--hard` uses (multi-JOIN / subquery / GROUP BY) and prints the actual failing question/gold/predicted query for anything `full` got wrong, so you can eyeball whether the shortfall concentrates in complex queries or is spread evenly.


In [ ]:
import json
from scripts.run_eval import _is_hard

report  = json.load(open("evaluation/results/phase18_sql_full.json"))
results = report["results"]

def bucket(r):
    return "hard" if _is_hard({"sql": r["gold"]}, "sql") else "easy"

buckets = {"hard": {"n": 0, "wrong": 0}, "easy": {"n": 0, "wrong": 0}}
failures = []
for r in results:
    full = r["ablations"]["full"]
    if not full["ex_eligible"]:
        continue
    b = bucket(r)
    buckets[b]["n"] += 1
    if full["ex"] == 0.0:
        buckets[b]["wrong"] += 1
        failures.append(r)

print(f"{'bucket':6} {'n':>4} {'EX':>8}")
for b, d in buckets.items():
    n, wrong = d["n"], d["wrong"]
    ex = (n - wrong) / n if n else None
    print(f"{b:6} {n:4} {f'{ex:.1%}' if ex is not None else 'n/a':>8}")

print(f"\n{len(failures)} misses out of {sum(d['n'] for d in buckets.values())} scored questions\n")
for r in failures:
    full = r["ablations"]["full"]
    print(f"[{bucket(r)}] {r['question']}  (db={r['db_name']})")
    print(f"    gold: {r['gold']}")
    print(f"    pred: {full['query']}")
    if full["pred_error"]:
        print(f"    error: {full['pred_error']}")
    print()

bucket    n       EX
hard     59    69.5%
easy     41    95.1%

20 misses out of 100 scored questions

[hard] Which countries have greater area than that of any country in Europe?  (db=world_1)
    gold: SELECT Name FROM country WHERE SurfaceArea  >  (SELECT min(SurfaceArea) FROM country WHERE Continent  =  "Europe")
    pred: SELECT Name FROM Country WHERE SurfaceArea  >  (SELECT max(SurfaceArea) FROM Country WHERE Continent  =  'Europe')

[hard] Find the model of the car whose weight is below the average weight.  (db=car_1)
    gold: SELECT T1.model FROM CAR_NAMES AS T1 JOIN CARS_DATA AS T2 ON T1.MakeId  =  T2.Id WHERE T2.Weight  <  (SELECT avg(Weight) FROM CARS_DATA)
    pred: SELECT Model FROM model_list EXCEPT SELECT T1.Model FROM model_list AS T1 JOIN car_names AS T2 ON T1.Model  =  T2.Model WHERE T2.Make LIKE "chevrolet%" INTERSECT SELECT T1.Model FROM model_list AS T1 JOIN car_names AS T2 ON T1.Model  =  T2.Model WHERE T2.Make LIKE "volkswagen%"

[hard] What is the total number

## 10. NoSQL track — smoke test

Requires the live `mongod` + loaded databases from section 4. Uses the **train** split (no held-out NoSQL set exists) — see the caveat at the top of this notebook.

In [ ]:
%%time
!python -m scripts.run_eval --track nosql --data Data/cot_data/nosql_cot_train.json \
    --n 10 --ablation all --out evaluation/results/phase18_nosql_smoke.json

Evaluating 10 questions x 4 ablation(s) = 30 generation passes (3 per question — configurations sharing generator inputs are batched).
[!] --data path contains 'train'. The Generator was fine-tuned on the train CoT files, so EX there reflects memorization, not generalization.
[1/10] show the titles, and authors or editors for all books made after the year 1989.  (db=culture_company)
Running on: cuda
Loading weights: 100% 391/391 [00:00<00:00, 2829.89it/s]
Loaded corpus: 5697 entries
Pre-computing corpus embeddings ...
pre tokenize: 100% 23/23 [00:00<00:00, 198.08it/s]
Inference Embeddings: 100% 23/23 [00:00<00:00, 31.97it/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:03<00:00, 89.17it/s] 
    full              ex=1.0  exact=True
    no_schema_linker  ex=1.0  exact=True
    no_sar            ex=0.0  exact=False
    no_posg           ex=1.0  exact=True
[2/10] Give the names of people who did not participate in the candidate election

## 11. NoSQL track — full ablation sweep

Target: **>60% EX** for `full`. Remember this number is on the training split, so report it in Phase 19 as an upper bound / memorization check, not a generalization result.

In [10]:
%%time
!python -m scripts.run_eval --track nosql --data Data/cot_data/nosql_cot_train.json \
    --n 100 --ablation all --out evaluation/results/phase18_nosql_full.json

Evaluating 100 questions x 4 ablation(s) = 300 generation passes (3 per question — configurations sharing generator inputs are batched).
[!] --data path contains 'train'. The Generator was fine-tuned on the train CoT files, so EX there reflects memorization, not generalization.
[1/100] show the titles, and authors or editors for all books made after the year 1989.  (db=culture_company)
Running on: cuda
config.json: 100% 779/779 [00:00<00:00, 4.27MB/s]
tokenizer_config.json: 100% 366/366 [00:00<00:00, 2.11MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 19.2MB/s]
tokenizer.json: 100% 711k/711k [00:00<00:00, 73.2MB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 785kB/s]

model.safetensors: downloading bytes:  23% 313M/1.34G [00:01<00:03, 326MB/s, 25.8MB/s  ]
model.safetensors: downloading bytes:  33% 436M/1.34G [00:01<00:02, 438MB/s, 34.8MB/s  ]
model.safetensors: downloading bytes:  50% 673M/1.34G [00:02<00:01, 547MB/s, 56.3MB/s  ]
model.safetensors: downloading bytes:  59% 794M/1.

### 12. Free the GPU when you're done

Units keep burning as long as the runtime stays connected, even idle. Run this once you've confirmed the reports landed on Drive above.

In [11]:
from google.colab import runtime
runtime.unassign()